# Task 01：从零理解多臂老虎机

本 Notebook 是可执行讲义。核心算法只存在于 `src/`，这里负责把公式、代码和实验串起来。

学习目标：区分真实价值、估计价值与采样奖励；掌握增量更新；比较 ε-greedy、UCB 和 Thompson Sampling；理解 pseudo-regret 与非平稳估计。

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "task-01-bandit", Path.cwd().parent]
TASK_DIR = next((p.resolve() for p in candidates if (p / "src" / "bandits.py").is_file()), None)
assert TASK_DIR is not None, "请从仓库根、任务根或 notebooks/ 目录启动 Notebook"
if str(TASK_DIR) not in sys.path:
    sys.path.insert(0, str(TASK_DIR))

import matplotlib.pyplot as plt
import numpy as np
from src.agents import EpsilonGreedyAgent, ThompsonSamplingAgent, UCBAgent, incremental_update
from src.bandits import BernoulliBandit, NonStationaryGaussianBandit
from src.experiment import plot_results, run_experiment

plt.style.use("seaborn-v0_8-whitegrid")
print("Task directory:", TASK_DIR)
print("NumPy version:", np.__version__)

## 1. 环境：真实价值不等于单次奖励

Bernoulli 臂的真实价值是成功概率 $q_*(a)=p_a$。但一次 `pull` 只能看到 0 或 1。算法必须从带噪声样本反推哪个动作最好。

In [ ]:
env = BernoulliBandit([0.1, 0.5, 0.9], seed=42)
samples = np.array([[env.pull(action) for _ in range(20)] for action in range(env.n_arms)])
print("真实期望奖励:", env.expected_rewards)
print("20 次样本均值:", samples.mean(axis=1))
print("最优动作:", env.optimal_arm)
print("action=0 的单步 pseudo-regret:", env.regret(0))

## 2. 增量更新

样本平均与固定步长共用 $Q \leftarrow Q + \alpha(R-Q)$。sample-average 使用 $\alpha=1/N(a)$；constant step-size 使用固定 $\alpha$。

In [ ]:
rewards = [2.0, 4.0, 0.0]
sample_average = constant_step = 0.0
print("count | reward | sample-average | alpha=0.5")
for count, reward in enumerate(rewards, start=1):
    sample_average = incremental_update(sample_average, reward, count=count)
    constant_step = incremental_update(constant_step, reward, count=count, step_size=0.5)
    print(f"{count:5d} | {reward:6.1f} | {sample_average:14.3f} | {constant_step:9.3f}")

## 3. ε-greedy：探索率消融

只改变 ε，其他环境和实验预算不变。多次独立实验逐时间步求均值，避免用单条幸运轨迹下结论。

In [ ]:
probabilities = [0.1, 0.3, 0.5, 0.7, 0.9]
epsilon_results = {}
for epsilon in [0.0, 0.01, 0.1, 0.3]:
    epsilon_results[f"epsilon={epsilon}"] = run_experiment(
        lambda seed: BernoulliBandit(probabilities, seed=seed),
        lambda seed, eps=epsilon: EpsilonGreedyAgent(len(probabilities), epsilon=eps, seed=seed),
        n_runs=60, n_steps=400, seed=10,
    )
plot_results(epsilon_results)
plt.show()

**观察：** 纯 greedy 可能因初期噪声锁定次优臂；ε 太大则后期仍持续随机探索。不存在脱离 horizon 和 reward gap 的万能 ε。

## 4. 三种策略主实验

主实验使用 Bernoulli 环境，因为三种策略都与它兼容。UCB 使用显式置信 bonus；Thompson 从 Beta 后验采样。

In [ ]:
main_results = {
    "epsilon-greedy": run_experiment(
        lambda seed: BernoulliBandit(probabilities, seed=seed),
        lambda seed: EpsilonGreedyAgent(len(probabilities), epsilon=0.1, seed=seed),
        n_runs=100, n_steps=600, seed=42),
    "ucb": run_experiment(
        lambda seed: BernoulliBandit(probabilities, seed=seed),
        lambda seed: UCBAgent(len(probabilities), c=2.0, seed=seed),
        n_runs=100, n_steps=600, seed=42),
    "thompson": run_experiment(
        lambda seed: BernoulliBandit(probabilities, seed=seed),
        lambda seed: ThompsonSamplingAgent(len(probabilities), seed=seed),
        n_runs=100, n_steps=600, seed=42),
}
for name, result in main_results.items():
    print(name, result.summary())
plot_results(main_results)
plt.show()

### Thompson 后验不是一个点

一个动作经历 8 次成功、2 次失败后，均匀先验变为 `Beta(9,3)`。分布宽度表达不确定性，数据增加后逐渐收窄。

In [ ]:
import math
alpha, beta = 9.0, 3.0
x = np.linspace(0.001, 0.999, 400)
log_b = math.lgamma(alpha) + math.lgamma(beta) - math.lgamma(alpha + beta)
density = np.exp((alpha - 1) * np.log(x) + (beta - 1) * np.log(1 - x) - log_b)
plt.figure(figsize=(7, 3.5))
plt.plot(x, density)
plt.axvline(alpha / (alpha + beta), color="tab:red", linestyle="--", label="posterior mean")
plt.xlabel("Bernoulli success probability")
plt.ylabel("density")
plt.title("Beta(9, 3) posterior")
plt.legend()
plt.show()

## 5. 非平稳环境：样本平均 vs 固定步长

动作价值每一步随机游走。样本平均的有效步长越来越小；固定 `α=0.1` 始终关注新证据。两组算法只改变估计器。

In [ ]:
def drifting_env(seed):
    return NonStationaryGaussianBandit(np.zeros(10), std=1.0, random_walk_std=0.03, seed=seed)

nonstationary_results = {
    "sample-average": run_experiment(
        drifting_env, lambda seed: EpsilonGreedyAgent(10, epsilon=0.1, step_size=None, seed=seed),
        n_runs=60, n_steps=800, seed=7),
    "constant alpha=0.1": run_experiment(
        drifting_env, lambda seed: EpsilonGreedyAgent(10, epsilon=0.1, step_size=0.1, seed=seed),
        n_runs=60, n_steps=800, seed=7),
}
for name, result in nonstationary_results.items():
    print(name, result.summary())
plot_results(nonstationary_results)
plt.show()

## 6. 兼容边界与思考题

ε-greedy 和 UCB 支持 Bernoulli/Gaussian；本仓库的 Thompson 是 Beta-Bernoulli 版本，只接受 0/1。

<details><summary>为什么非平稳环境要在 pull 前计算 pseudo-regret？</summary>`pull` 后均值会漂移。先记录 gap 才能保证动作与 q_t 属于同一时刻。</details>

<details><summary>为什么单条轨迹不能证明 A 优于 B？</summary>环境奖励和动作选择都有随机性，需要多 seed 均值、方差或置信区间。</details>

完整自检：`python eval/run.py`。修改任意公式后都应重新运行测试和 Notebook。